# Streaming Events

## Objective

Process ride events using Spark Structured Streaming.

### Learning Goals

- Structured Streaming
- Streaming DataFrames
- Triggers
- Checkpointing
- Stream-to-Delta pattern

### Output

- workspace.default.bronze_ride_events

## Why Streaming?

Traditional batch processing works on static datasets.

Streaming processes data continuously as it arrives from sources such as:

- Kafka
- Event Hubs
- Kinesis
- Auto Loader
- IoT Devices

In [0]:
stream_df = (
    spark.readStream
         .format("rate")
         .option("rowsPerSecond", 1000)
         .load()
)

## Review Streaming Schema

The rate source generates two columns:

- timestamp
- value

These records will be transformed into ride events.

In [0]:
stream_df.printSchema()

## Create Ride Events

Transform generated values into simulated ride lifecycle events.

In [0]:
from pyspark.sql.functions import (
    col,
    expr
)

ride_events_df = (
    stream_df
    .withColumn(
        "ride_id",
        col("value")
    )
    .withColumn(
        "event_type",
        expr("""
        CASE
            WHEN value % 3 = 0 THEN 'REQUESTED'
            WHEN value % 3 = 1 THEN 'STARTED'
            ELSE 'COMPLETED'
        END
        """)
    )
)

## Streaming DataFrames

Unlike a traditional DataFrame, a Streaming DataFrame represents an unbounded stream of data.

New records continuously arrive and are processed in micro-batches.

## Checkpointing

Structured Streaming uses checkpoints to store:

- Progress information
- Offsets
- Recovery state

This allows streaming jobs to recover after failures.

Databricks Free Edition requires explicit checkpoint locations.

In [0]:
checkpoint_path = (
    "/Volumes/workspace/default/raw_data/"
    "checkpoints/ride_events"
)

## Trigger Mode

Databricks Free Edition does not support continuously running ProcessingTime triggers on this compute type.

AvailableNow is used to demonstrate micro-batch execution.

In [0]:
(
    ride_events_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .toTable(
        "workspace.default.bronze_ride_events"
    )
)

## Validation

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_ride_events
LIMIT 20

In [0]:
%sql
SELECT
    event_type,
    COUNT(*) AS total_events
FROM workspace.default.bronze_ride_events
GROUP BY event_type
ORDER BY total_events DESC

## Streaming Concepts Covered

### Structured Streaming

✅ readStream

✅ writeStream

✅ Streaming DataFrames

✅ Checkpointing

✅ Trigger Modes

✅ Stream-to-Delta Pattern

### Free Edition Limitation

Continuous ProcessingTime triggers are not supported on this compute type.

AvailableNow trigger was used to execute streaming micro-batches.